In [31]:
import pandas as pd          
import numpy as np           
import matplotlib.pyplot as plt   
import seaborn as sns       


In [32]:
# Read the dataset from the current folder
df = pd.read_csv("vgsales_input.csv")

# Show first rows to check if loading worked
df.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [33]:
# Check basic information about the dataset
df.info()

# Check first descriptive statistics
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  object 
 2   Platform      16598 non-null  object 
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  object 
 5   Publisher     16540 non-null  object 
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), object(4)
memory usage: 1.4+ MB


,Rank,Year,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
count,16598.000000,16327.000000,16598.000000,16598.000000,16598.000000,16598.000000,16598.000000
mean,8300.605254,2006.406443,0.264667,0.146652,0.077782,0.048063,0.537441
std,4791.853933,5.828981,0.816683,0.505351,0.309291,0.188588,1.555028
min,1.000000,1980.000000,0.000000,0.000000,0.000000,0.000000,0.010000
25%,4151.250000,2003.000000,0.000000,0.000000,0.000000,0.000000,0.060000
50%,8300.500000,2007.000000,0.080000,0.020000,0.000000,0.010000,0.170000
75%,12449.750000,2010.000000,0.240000,0.110000,0.040000,0.040000,0.470000
max,16600.000000,2020.000000,41.490000,29.020000,10.220000,10.570000,82.740000


# Data Cleaning

# Missing values

In [34]:
# Check missing values per column
df.isnull().sum()

Rank              0
Name              0
Platform          0
Year            271
Genre             0
Publisher        58
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64

# Deal with them (delete everything below 1 million, research the rest)

In [35]:
# Filter rows where Publisher is missing (NaN)
missing_publisher = df[df["Publisher"].isna()]

# Sort by Global Sales descending
missing_publisher = missing_publisher.sort_values(by="Global_Sales", ascending=False)

# Display all rows
missing_publisher

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
470,471,wwe Smackdown vs. Raw 2006,PS2,NaN,Fighting,NaN,1.57,1.02,0.00,0.41,3.00
1303,1305,Triple Play 99,PS,NaN,Sports,NaN,0.81,0.55,0.00,0.10,1.46
1662,1664,Shrek / Shrek 2 2-in-1 Gameboy Advance Video,GBA,2007.0,Misc,NaN,0.87,0.32,0.00,0.02,1.21
2222,2224,Bentley's Hackpack,GBA,2005.0,Misc,NaN,0.67,0.25,0.00,0.02,0.93
3159,3161,Nicktoons Collection: Game Boy Advance Video V...,GBA,2004.0,Misc,NaN,0.46,0.17,0.00,0.01,0.64
3166,3168,SpongeBob SquarePants: Game Boy Advance Video ...,GBA,2004.0,Misc,NaN,0.46,0.17,0.00,0.01,0.64
3766,3768,SpongeBob SquarePants: Game Boy Advance Video ...,GBA,2004.0,Misc,NaN,0.38,0.14,0.00,0.01,0.53
4145,4147,Sonic the Hedgehog,PS3,NaN,Platform,NaN,0.00,0.48,0.00,0.00,0.48
4526,4528,The Fairly Odd Parents: Game Boy Advance Video...,GBA,2004.0,Misc,NaN,0.31,0.11,0.00,0.01,0.43
4635,4637,The Fairly Odd Parents: Game Boy Advance Video...,GBA,2004.0,Misc,NaN,0.30,0.11,0.00,0.01,0.42


# Delete all missing publisher with less than 1 mio global sales

In [36]:
# Remove rows where Publisher is missing AND Global_Sales < 1
df = df[~(
    df["Publisher"].isna() & 
    (df["Global_Sales"] < 1)
)]

In [37]:
# Check remaining missing publishers
df[df["Publisher"].isna()].sort_values(by="Global_Sales", ascending=False)

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
470,471,wwe Smackdown vs. Raw 2006,PS2,NaN,Fighting,NaN,1.57,1.02,0.0,0.41,3.00
1303,1305,Triple Play 99,PS,NaN,Sports,NaN,0.81,0.55,0.0,0.10,1.46
1662,1664,Shrek / Shrek 2 2-in-1 Gameboy Advance Video,GBA,2007.0,Misc,NaN,0.87,0.32,0.0,0.02,1.21


# manually plug in the publishers

In [38]:
# Update publisher for WWE Smackdown vs. Raw 2006 (PS2)
df.loc[
    (df["Name"] == "wwe Smackdown vs. Raw 2006") &
    (df["Platform"] == "PS2"),
    "Publisher"
] = "THQ"


# Update publisher for Triple Play 99 (PS)
df.loc[
    (df["Name"] == "Triple Play 99") &
    (df["Platform"] == "PS"),
    "Publisher"
] = "Electronic Arts"


# Update publisher for Shrek / Shrek 2 2-in-1 Gameboy Advance Video (GBA)
df.loc[
    (df["Name"] == "Shrek / Shrek 2 2-in-1 Gameboy Advance Video") &
    (df["Platform"] == "GBA"),
    "Publisher"
] = "Majesco Entertainment"

In [39]:
# Check if publishers were updated correctly
df.loc[
    df["Name"].isin([
        "wwe Smackdown vs. Raw 2006",
        "Triple Play 99",
        "Shrek / Shrek 2 2-in-1 Gameboy Advance Video"
    ]),
    ["Name", "Platform", "Publisher"]
]

,Name,Platform,Publisher
470,wwe Smackdown vs. Raw 2006,PS2,THQ
1303,Triple Play 99,PS,Electronic Arts
1662,Shrek / Shrek 2 2-in-1 Gameboy Advance Video,GBA,Majesco Entertainment


# now for years

In [40]:
# Filter rows where Year is missing
missing_year = df[df["Year"].isna()]

# Sort by Global Sales descending
missing_year = missing_year.sort_values(by="Global_Sales", ascending=False)

# Show full list
pd.set_option("display.max_rows", None)
missing_year

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
179,180,Madden NFL 2004,PS2,NaN,Sports,Electronic Arts,4.26,0.26,0.01,0.71,5.23
377,378,FIFA Soccer 2004,PS2,NaN,Sports,Electronic Arts,0.59,2.36,0.04,0.51,3.49
431,432,LEGO Batman: The Videogame,Wii,NaN,Action,Warner Bros. Interactive Entertainment,1.86,1.02,0.00,0.29,3.17
470,471,wwe Smackdown vs. Raw 2006,PS2,NaN,Fighting,THQ,1.57,1.02,0.00,0.41,3.00
607,608,Space Invaders,2600,NaN,Shooter,Atari,2.36,0.14,0.00,0.03,2.53
624,625,Rock Band,X360,NaN,Misc,Electronic Arts,1.93,0.34,0.00,0.21,2.48
649,650,Frogger's Adventures: Temple of the Frog,GBA,NaN,Adventure,Konami Digital Entertainment,2.15,0.18,0.00,0.07,2.39
652,653,LEGO Indiana Jones: The Original Adventures,Wii,NaN,Action,LucasArts,1.54,0.63,0.00,0.22,2.39
711,713,Call of Duty 3,Wii,NaN,Shooter,Activision,1.19,0.84,0.00,0.23,2.26
782,784,Rock Band,Wii,NaN,Misc,MTV Games,1.35,0.56,0.00,0.20,2.11


In [41]:
# Remove rows where Year is missing AND Global_Sales < 1
df = df[~(
    df["Year"].isna() &
    (df["Global_Sales"] < 1)
)]

In [42]:
# Check remaining missing Year values
df[df["Year"].isna()].sort_values(by="Global_Sales", ascending=False)

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
179,180,Madden NFL 2004,PS2,NaN,Sports,Electronic Arts,4.26,0.26,0.01,0.71,5.23
377,378,FIFA Soccer 2004,PS2,NaN,Sports,Electronic Arts,0.59,2.36,0.04,0.51,3.49
431,432,LEGO Batman: The Videogame,Wii,NaN,Action,Warner Bros. Interactive Entertainment,1.86,1.02,0.00,0.29,3.17
470,471,wwe Smackdown vs. Raw 2006,PS2,NaN,Fighting,THQ,1.57,1.02,0.00,0.41,3.00
607,608,Space Invaders,2600,NaN,Shooter,Atari,2.36,0.14,0.00,0.03,2.53
624,625,Rock Band,X360,NaN,Misc,Electronic Arts,1.93,0.34,0.00,0.21,2.48
649,650,Frogger's Adventures: Temple of the Frog,GBA,NaN,Adventure,Konami Digital Entertainment,2.15,0.18,0.00,0.07,2.39
652,653,LEGO Indiana Jones: The Original Adventures,Wii,NaN,Action,LucasArts,1.54,0.63,0.00,0.22,2.39
711,713,Call of Duty 3,Wii,NaN,Shooter,Activision,1.19,0.84,0.00,0.23,2.26
782,784,Rock Band,Wii,NaN,Misc,MTV Games,1.35,0.56,0.00,0.20,2.11


In [43]:
# Update missing years based on Rank (unique identifier here)

rank_to_year = {
    180: 2003,   # Madden NFL 2004 (PS2)
    378: 2003,  # FIFA Soccer 2004 (PS2) -> TODO: add the year you researched
    432: 2008,   # LEGO Batman: The Videogame (Wii)
    471: 2005,   # wwe Smackdown vs. Raw 2006 (PS2)
    608: 1980,   # Space Invaders (2600)
    625: 2007,   # Rock Band (X360)
    650: 2001,   # Frogger's Adventures: Temple of the Frog (GBA)
    653: 2008,   # LEGO Indiana Jones: The Original Adventures
    713: 2006,   # Call of Duty 3
    784: 2008,   # Rock Band (second entry)
    1128: 2012   # Call of Duty: Black Ops
}

for rank, year in rank_to_year.items():
    df.loc[df["Rank"] == rank, "Year"] = year

In [44]:
# Check remaining missing Year values
df[df["Year"].isna()].sort_values(by="Global_Sales", ascending=False)

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
1133,1135,Rock Band,PS3,NaN,Misc,Electronic Arts,0.99,0.41,0.0,0.22,1.62
1303,1305,Triple Play 99,PS,NaN,Sports,Electronic Arts,0.81,0.55,0.0,0.10,1.46
1433,1435,LEGO Harry Potter: Years 5-7,Wii,NaN,Action,Warner Bros. Interactive Entertainment,0.76,0.47,0.0,0.13,1.36
1498,1500,LEGO Batman: The Videogame,PSP,NaN,Action,Warner Bros. Interactive Entertainment,0.57,0.46,0.0,0.28,1.32
1513,1515,Adventure,2600,NaN,Adventure,Atari,1.21,0.08,0.0,0.01,1.30
1585,1587,Combat,2600,NaN,Action,Atari,1.17,0.07,0.0,0.01,1.25
1649,1651,NASCAR Thunder 2003,PS2,NaN,Racing,Unknown,0.60,0.46,0.0,0.16,1.22
1697,1699,Hitman 2: Silent Assassin,XB,NaN,Action,Eidos Interactive,0.76,0.38,0.0,0.05,1.19
1837,1839,Rock Band,PS2,NaN,Misc,Electronic Arts,0.71,0.06,0.0,0.35,1.11
1990,1992,Legacy of Kain: Soul Reaver,PS,NaN,Action,Eidos Interactive,0.58,0.40,0.0,0.07,1.04


In [45]:
# Update missing years based on Rank (unique identifier here)

rank_to_year = {
    1135: 2008,  # Rock Band (PS3)
    1305: 1998,  # Triple Play 99 (PS)
    1435: 2011,  # LEGO Harry Potter: Years 5-7 (Wii)
    1500: 2008,  # LEGO Batman: The Videogame (PSP)
    1515: 1980,  # Adventure (2600)
    1587: 1980,  # Combat (2600)
    1651: 2002,  # NASCAR Thunder 2003 (PS2)
    1699: 2002,  # Hitman 2: Silent Assassin (XB)
    1839: 2007,  # Rock Band (PS2)
    1992: 2012,  # Legacy of Kain: Soul Reaver (PS)
    2021: 1997   # Donkey Kong Land III (GB)
}

for rank, year in rank_to_year.items():
    df.loc[df["Rank"] == rank, "Year"] = year

In [46]:
# Check missing values per column
df.isnull().sum()

Rank            0
Name            0
Platform        0
Year            0
Genre           0
Publisher       0
NA_Sales        0
EU_Sales        0
JP_Sales        0
Other_Sales     0
Global_Sales    0
dtype: int64

# Duplicates

In [47]:
# Create list of all columns except "Rank"
cols_without_rank = df.columns.drop("Rank")

In [48]:
# Count such duplicates
df.duplicated(subset=cols_without_rank).sum()

np.int64(0)

In [49]:
# Show all rows that are duplicates based on Name, Platform and years
duplicates = df[df.duplicated(subset=["Name", "Platform", "Year"], keep=False)]

duplicates.sort_values(by=["Name", "Platform"])

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
603,604,Madden NFL 13,PS3,2012.0,Sports,Electronic Arts,2.11,0.23,0.0,0.22,2.56
16127,16130,Madden NFL 13,PS3,2012.0,Sports,Electronic Arts,0.00,0.01,0.0,0.00,0.01


# Combine the obvious duplicates

In [50]:
# Define the two ranks
rank_keep = 604
rank_remove = 16130

# Define sales columns
sales_cols = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"]

# Add sales from duplicate row to the row we keep
df.loc[df["Rank"] == rank_keep, sales_cols] = (
    df.loc[df["Rank"] == rank_keep, sales_cols].values +
    df.loc[df["Rank"] == rank_remove, sales_cols].values
)

# Remove the duplicate row
df = df[df["Rank"] != rank_remove]

In [51]:
df = df.drop(columns=["Rank"])

In [52]:
df[df["Name"] == "Madden NFL 13"][["Name", "Platform", "Global_Sales"]]

,Name,Platform,Global_Sales
506,Madden NFL 13,X360,2.86
603,Madden NFL 13,PS3,2.57
3730,Madden NFL 13,Wii,0.54
5588,Madden NFL 13,PSV,0.32
6792,Madden NFL 13,WiiU,0.24


# There was one year 2020, which is wrong. Manually checking showed that the release year is 2009

In [53]:
# Show all rows where Year is 2020
df[df["Year"] == 2020]

,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
5957,Imagine: Makeup Artist,DS,2020.0,Simulation,Ubisoft,0.27,0.0,0.0,0.02,0.29


In [54]:
# Replace Year 2020 with 2009
df.loc[df["Year"] == 2020, "Year"] = 2009

In [55]:
# Verify no 2020 values remain
(df["Year"] == 2020).sum()

np.int64(0)

# Cleaning finished, export final dataset

In [56]:
# Save dataframe as Excel file
df.to_excel("Games_final.xlsx", index=False)

In [58]:
# Read final dataset
df = pd.read_excel("Games_final.xlsx")

# Quick check
df.head()

,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


# Get descriptive statistics and dataset overview

In [62]:
# Create structure table
structure_df = pd.DataFrame({
    "Feature / Variable": df.columns,
    "Data type": df.dtypes.values,
    "Number of unique values": [df[col].nunique() for col in df.columns],
    "Example values": [df[col].dropna().unique()[:3] for col in df.columns]
})

# Display structure table
structure_df

,Feature / Variable,Data type,Number of unique values,Example values
0,Name,object,11334,"[Wii Sports, Super Mario Bros., Mario Kart Wii]"
1,Platform,object,31,"[Wii, NES, GB]"
2,Year,int64,38,"[2006, 1985, 2008]"
3,Genre,object,12,"[Sports, Platform, Racing]"
4,Publisher,object,576,"[Nintendo, Microsoft Game Studios, Take-Two In..."
5,NA_Sales,float64,409,"[41.49, 29.08, 15.85]"
6,EU_Sales,float64,305,"[29.02, 3.58, 12.88]"
7,JP_Sales,float64,244,"[3.77, 6.81, 3.79]"
8,Other_Sales,float64,157,"[8.46, 0.77, 3.31]"
9,Global_Sales,float64,623,"[82.74, 40.24, 35.82]"


In [63]:
# Select numeric columns
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

# Calculate descriptive statistics
desc_stats = df[numeric_cols].describe().T

# Add variance
desc_stats["Variance"] = df[numeric_cols].var()

# Display numeric statistics table
desc_stats

,count,mean,std,min,25%,50%,75%,max,Variance
Year,16313.0,2006.398762,5.840202,1980.00,2003.00,2007.00,2010.00,2017.00,34.107961
NA_Sales,16313.0,0.267072,0.823320,0.00,0.00,0.08,0.24,41.49,0.677856
EU_Sales,16313.0,0.148276,0.509470,0.00,0.00,0.02,0.11,29.02,0.259560
JP_Sales,16313.0,0.078730,0.311682,0.00,0.00,0.00,0.04,10.22,0.097146
Other_Sales,16313.0,0.048640,0.190146,0.00,0.00,0.01,0.04,10.57,0.036155
Global_Sales,16313.0,0.542988,1.567676,0.01,0.06,0.17,0.48,82.74,2.457608
